# Lorentz boost of the field tensor

*em · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/field_tensor_boost)

In relativity $\mathbf E$ and $\mathbf B$ are not separate vectors but the six components of one antisymmetric $4\times 4$ tensor $F^{\mu\nu}$. A Lorentz boost transforms it with one $\Lambda$ per index, $F' = \Lambda F\Lambda^{\mathsf T}$, written as a single einsum. Sweeping the boost speed shows electric and magnetic fields mixing into each other while $F_{\mu\nu}F^{\mu\nu}$ and $\mathbf E\cdot\mathbf B$ stay fixed.

**Vector form**

$$\mathbf E'_\parallel = \mathbf E_\parallel,\;\; \mathbf E'_\perp = \gamma(\mathbf E + \boldsymbol\beta\times\mathbf B)_\perp,\;\; \mathbf B'_\perp = \gamma(\mathbf B - \boldsymbol\beta\times\mathbf E)_\perp$$

**Index (tensor) form**

$$F'^{\mu\nu} = \Lambda^{\mu}{}_{\alpha}\,\Lambda^{\nu}{}_{\beta}\,F^{\alpha\beta},\qquad F^{0i} = -E_i,\;\; F^{ij} = -\varepsilon_{ijk}B_k$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $F^{\mu\nu}$ | `field_tensor(E, B, eps)` | (4, 4) | antisymmetric field tensor, $\mu, \nu \in \{t, x, y, z\}$ |
| $\eta_{\mu\nu}$ | `metric()` | (4, 4) | Minkowski metric $\operatorname{diag}(+1, -1, -1, -1)$, $c = 1$ |
| $\Lambda^{\mu}{}_{\alpha}$ | `boost(beta)` | (S, 4, 4) | boost along $x$ for each of $S$ speeds $\beta$ |
| $\Lambda^{\mu}{}_{\alpha}\Lambda^{\nu}{}_{\beta}F^{\alpha\beta}$ | `einsum('sma,snb,ab->smn', Lam, Lam, F)` | (S, 4, 4) | repeated $\alpha$ and $\beta$ ⇒ summed: one $\Lambda$ per index of $F$ |
| $F_{\mu\nu}$ | `einsum('ma,nb,...ab->...mn', eta, eta, F)` | (…, 4, 4) | indices lowered with the metric |
| $F_{\mu\nu}F^{\mu\nu}$ | `einsum('...mn,...mn->...', F_low, F)` | (S,) | invariant $2(B^2 - E^2)$ |
| $F_{\mu\nu}\tilde F^{\mu\nu}$ | `pseudo_invariant(F, eta, eps4)` | (S,) | invariant $-4\,\mathbf E\cdot\mathbf B$, using the dual tensor $\tfrac12\varepsilon^{\mu\nu\rho\sigma}F_{\rho\sigma}$ |

Units with $c = 1$, metric signature $(+, -, -, -)$. The boost is along $x$, so $E_x$ and $B_x$ never change; only the transverse components mix. For a pure electric field ($\mathbf B = 0$) the moving observer sees a magnetic field $\mathbf B' = -\gamma\,\boldsymbol\beta\times\mathbf E$. Both Lorentz invariants stay constant along the whole sweep.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/field_tensor_boost/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve, tensors
from .equation import field_tensor


def run(params: dict, out_dir: Path) -> Path:
    E, B = tensors.fields(params)  # (3,), (3,)
    F = field_tensor(E, B, tensors.levi_civita(3))  # (4, 4)   tiny tensors: CPU float64

    sweep = solve.solve(F, params["beta_max"])

    for key in ["E", "B"]:
        print(f"{key} at beta = 0: {sweep[key][0].tolist()}  ->  at beta_max: {sweep[key][-1].numpy().round(3)}")
    print(f"F_mn F^mn = {sweep['FF'][0]:.4f} -> {sweep['FF'][-1]:.4f}   (invariant)")
    return plot.animate(sweep, out_dir / "field_tensor_boost.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/field_tensor_boost/equation.py
r"""Physics → tensor translation.

E and B are the six independent components of the antisymmetric field tensor (c = 1,
metric η = diag(+1, −1, −1, −1)):

    F^{0i} = -E_i,     F^{i0} = E_i,     F^{ij} = -\varepsilon_{ijk} B_k

             ⎡  0   -E_x  -E_y  -E_z ⎤
    F^{μν} = ⎢ E_x    0   -B_z   B_y ⎥
             ⎢ E_y   B_z    0   -B_x ⎥
             ⎣ E_z  -B_y   B_x    0  ⎦

A tensor with two upper indices transforms with one Λ per index:

    F'^{\mu\nu} = \Lambda^{\mu}{}_{\alpha} \Lambda^{\nu}{}_{\beta} F^{\alpha\beta}

α and β are repeated, so both are summed — einsum('ma,nb,ab->mn'), i.e. Λ F Λᵀ.
Contracting all indices gives numbers every observer agrees on (Lorentz invariants):

    F_{\mu\nu} F^{\mu\nu} = 2 (B^2 - E^2),     F_{\mu\nu} \tilde F^{\mu\nu} = -4\, \mathbf E \cdot \mathbf B

with indices lowered by the metric, F_{\mu\nu} = \eta_{\mu\alpha} \eta_{\nu\beta} F^{\alpha\beta},
and the dual tensor \tilde F^{\mu\nu} = ½ \varepsilon^{\mu\nu\rho\sigma} F_{\rho\sigma}.
"""

import torch


def field_tensor(E: torch.Tensor, B: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""F^{μν} of shape (4, 4) from E_i, B_k (3,) and the 3D Levi-Civita ε_{ijk} (3, 3, 3)."""
    F = torch.zeros(4, 4, dtype=E.dtype)
    F[0, 1:] = -E  # F^{0i} = -E_i
    F[1:, 0] = E  # F^{i0} = +E_i   (antisymmetric)
    F[1:, 1:] = -torch.einsum("ijk,k->ij", eps, B)  # F^{ij} = -ε_ijk B_k
    return F


def electric(F: torch.Tensor) -> torch.Tensor:
    """E_i = F^{i0}.   (..., 4, 4) → (..., 3)"""
    return F[..., 1:, 0]


def magnetic(F: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""B_i = -½ ε_{ijk} F^{jk}.   (..., 4, 4) → (..., 3)"""
    return -0.5 * torch.einsum("ijk,...jk->...i", eps, F[..., 1:, 1:])


def transform(Lam: torch.Tensor, F: torch.Tensor) -> torch.Tensor:
    r"""F'^{μν} = Λ^μ_α Λ^ν_β F^{αβ} for a batch of S boosts.   Lam: (S, 4, 4), F: (4, 4) → (S, 4, 4)"""
    return torch.einsum("sma,snb,ab->smn", Lam, Lam, F)


def lower(F: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""F_{μν} = η_{μα} η_{νβ} F^{αβ}.   (..., 4, 4) → (..., 4, 4)"""
    return torch.einsum("ma,nb,...ab->...mn", eta, eta, F)


def invariant(F: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""F_{μν} F^{μν} = 2(B² − E²): every index is summed, a scalar per boost.   → (...)"""
    return torch.einsum("...mn,...mn->...", lower(F, eta), F)


def pseudo_invariant(F: torch.Tensor, eta: torch.Tensor, eps4: torch.Tensor) -> torch.Tensor:
    r"""F_{μν} \tilde F^{μν} = −4 E·B with the dual \tilde F^{μν} = ½ ε^{μνρσ} F_{ρσ}.   → (...)"""
    F_low = lower(F, eta)
    dual = 0.5 * torch.einsum("mnrs,...rs->...mn", eps4, F_low)
    return torch.einsum("...mn,...mn->...", F_low, dual)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/field_tensor_boost/tensors.py
"""Build the concrete tensors: metric, Levi-Civita symbols, fields and boost matrices."""

import itertools

import torch

DTYPE = torch.float64


def permutation_sign(p: tuple[int, ...]) -> float:
    """+1 for an even permutation, −1 for an odd one (count the pairs that are out of order)."""
    inversions = sum(p[i] > p[j] for i in range(len(p)) for j in range(i + 1, len(p)))
    return -1.0 if inversions % 2 else 1.0


def levi_civita(dim: int) -> torch.Tensor:
    """ε with dim indices, shape (dim,) * dim: ε_{ijk} for dim = 3, ε^{μνρσ} (ε^{0123} = +1) for dim = 4."""
    eps = torch.zeros((dim,) * dim, dtype=DTYPE)
    for p in itertools.permutations(range(dim)):
        eps[p] = permutation_sign(p)
    return eps


def metric() -> torch.Tensor:
    """η_{μν} = diag(+1, −1, −1, −1) of shape (4, 4)."""
    return torch.diag(torch.tensor([1.0, -1.0, -1.0, -1.0], dtype=DTYPE))


def fields(params: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """E_i and B_i from the parameters, each of shape (3,)."""
    E = torch.tensor([params["Ex"], params["Ey"], params["Ez"]], dtype=DTYPE)
    B = torch.tensor([params["Bx"], params["By"], params["Bz"]], dtype=DTYPE)
    return E, B


def boost(beta: torch.Tensor) -> torch.Tensor:
    """Λ^μ_ν for a boost with speed β along x, for each β: (S,) → (S, 4, 4).

        ⎡  γ   -γβ  0  0 ⎤
    Λ = ⎢ -γβ   γ   0  0 ⎥        γ = 1 / √(1 − β²)
        ⎢  0    0   1  0 ⎥
        ⎣  0    0   0  1 ⎦
    """
    gamma = 1.0 / torch.sqrt(1.0 - beta**2)  # (S,)
    Lam = torch.eye(4, dtype=DTYPE).repeat(len(beta), 1, 1)  # (S, 4, 4)
    Lam[:, 0, 0] = gamma
    Lam[:, 1, 1] = gamma
    Lam[:, 0, 1] = -gamma * beta
    Lam[:, 1, 0] = -gamma * beta
    return Lam


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/field_tensor_boost/solve.py
"""Sweep the boost speed: transform F^{μν} for every β at once and read off E', B' and the invariants."""

import torch

from . import tensors
from .equation import electric, invariant, magnetic, pseudo_invariant, transform


def solve(F: torch.Tensor, beta_max: float, n_steps: int = 100) -> dict[str, torch.Tensor]:
    """Return a dict of tensors along the sweep β = 0 … beta_max (S = n_steps values).

    beta (S,), F (S, 4, 4), E (S, 3), B (S, 3), FF (S,) = F_{μν}F^{μν}, FFdual (S,) = F_{μν}F̃^{μν}
    """
    beta = torch.linspace(0.0, beta_max, n_steps, dtype=tensors.DTYPE)  # (S,)
    Lam = tensors.boost(beta)  # (S, 4, 4)
    F_boosted = transform(Lam, F)  # (S, 4, 4)  all S boosts in one einsum

    eta, eps3, eps4 = tensors.metric(), tensors.levi_civita(3), tensors.levi_civita(4)
    return {
        "beta": beta,
        "F": F_boosted,
        "E": electric(F_boosted),
        "B": magnetic(F_boosted, eps3),
        "FF": invariant(F_boosted, eta),
        "FFdual": pseudo_invariant(F_boosted, eta, eps4),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/field_tensor_boost/plot.py
"""Animate the sweep: F'^{μν} as an annotated heatmap next to bars for E' and B'."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

LABELS = ["t", "x", "y", "z"]


def animate(sweep: dict, path: Path) -> Path:
    beta = sweep["beta"].numpy()  # (S,)
    F = sweep["F"].numpy()  # (S, 4, 4)
    EB = np.concatenate([sweep["E"].numpy(), sweep["B"].numpy()], axis=1)  # (S, 6): E'x E'y E'z B'x B'y B'z
    FF, FFdual = sweep["FF"].numpy(), sweep["FFdual"].numpy()  # (S,), (S,)
    limit = max(np.abs(F).max(), 1e-6)

    fig, (ax_f, ax_v) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1, 1.1]})
    header = fig.suptitle("")

    # left: the 4x4 tensor, one annotated cell per component
    image = ax_f.imshow(F[0], cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax_f.set_xticks(range(4), [f"ν={s}" for s in LABELS])
    ax_f.set_yticks(range(4), [f"μ={s}" for s in LABELS])
    ax_f.grid(False)
    ax_f.set_title(r"$F'^{\mu\nu}$")
    cells = [[ax_f.text(n, m, "", ha="center", va="center", fontsize=9) for n in range(4)] for m in range(4)]

    # right: the six field components read back out of F'
    names = ["E'x", "E'y", "E'z", "B'x", "B'y", "B'z"]
    bars = ax_v.bar(names, EB[0], color=[ACCENT[0]] * 3 + [ACCENT[1]] * 3)
    top = max(np.abs(EB).max() * 1.15, 1e-6)
    ax_v.set_ylim(-top, top)
    ax_v.axhline(0, color="#6b7280", lw=0.8)
    ax_v.grid(axis="x", visible=False)
    ax_v.set_axisbelow(True)
    ax_v.set_title("fields seen by the moving observer")

    def update(step: int):
        image.set_data(F[step])
        for m in range(4):
            for n in range(4):
                value = F[step, m, n]
                cells[m][n].set_text(f"{value:+.2f}" if m != n else "0")
                cells[m][n].set_color("white" if abs(value) > 0.6 * limit else "black")
        for bar, height in zip(bars, EB[step]):
            bar.set_height(height)
        gamma = 1 / np.sqrt(1 - beta[step] ** 2)
        header.set_text(f"boost along x:  β = {beta[step]:.3f},  γ = {gamma:.2f}")
        # the two Lorentz invariants: the same for every β
        invariant = r"$F_{\mu\nu}F^{\mu\nu} = 2(B^2-E^2) = %+.3f$" % FF[step]
        pseudo = r"$\mathbf{E}\cdot\mathbf{B} = -\frac{1}{4}F_{\mu\nu}\tilde F^{\mu\nu} = %+.3f$" % (-FFdual[step] / 4)
        ax_v.set_xlabel(invariant + "\n" + pseudo)

    update(0)
    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(beta)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'Ex': 0.3,
    'Ey': 1.0,
    'Ez': 0.0,
    'Bx': 0.4,
    'By': 0.0,
    'Bz': 0.5,
    'beta_max': 0.95
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.field_tensor_boost")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.field_tensor_boost.main")
problem_dir = Path("problems/em/field_tensor_boost")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
